<a href="https://colab.research.google.com/github/Murimi-Kinyua/Introduction-to-Machine-Learning/blob/main/Week_3_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Designing My Machine Learning WorkFlow

My proposed Project Problem is Applied Machine Learning for Short-term Electricity Demand and Supply Forecasting.

The Data I presume I would need include:


1.   The data on what I am looking to predict:


*   System Load / Electricity Demand -> This is the total power consumption across the grid in a certain region.
*   Renewable Power Supply Generation -> This accounts the Solar and Wind Generation which we subtract from the total load to get the Net demand.


2.   Weather and Environmental Data


*   Wind Speed and Direction -> Governs Wind Turbine outputs and Heat loss in buildings in cold areas.
*   Global Horizontal Irradiance -> Cloud cover and Direct Solar Radiation dictate solar farm output and rooftop PV generation.


*   Dry-Bulb Temperature -> Affect air conditioning Machines (HVAC)
*   Dew Point / Relative Humidity -> High humidity increases cooling demand


3.   Temporal and Operation Metadata


*   Clock Features -> Hour of the day, Day of the week, Month of the Year.
*   Calender Events -> Public Holidays, Major local Events ie Human Operational Schedules.


*   Daylight Indicators -> Sunset and Sunrise Times brings different light demand curves.


3.   Historical Power Usage Data.





**Potential Input Features and Output Targets For my project.**

Input Features include:


1.   Historical Memory Data


*   Common Demand Lags -> e.g Last Hour, 2 hours ago, Same hour yesterday, Same hour last week.
*   Rolling Statistics -> Moving average over the past 24 hours. Moving Standard deviation (Captures demand volatility). Daily Peak and Trough baselines.




2.   Weather and Environmental Drivers Data. Dictates HVAC usage and Renewable Output.


*   Temperatures -> strongest non-linear driver for heating/cooling spikes.
*   Relative Humidity -> Dictates human discomfort and air conditioning load.


*   Global Horizontal Irradiance -> Direct Solar Radiation for Solar supply units.
*   Wind Speed -> For wind generation and buildings heat transfer.


3.   Temporal and Calender features. Since Human Behaviour follow specific diurnal, weekly and annual schedules that can be defined.


*   Day of the week, month of the year, hour of the day


Output Features include:


1.   primary demand target -> Total Grid Load or Electricity demand
2.   renewable supply targets -> Genaration output from solar and wind assets


3.   net demand target -> the actual load that traditional power plants must satisfy.



















### **Uploading Sample Test DataSets**

**1. Live Kenyan Weather and Solar Data from Open-Meteo API**

In [5]:
import pandas as pd
import requests

# Fetch hourly climate data for Kenya directly into a panda dataframe
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": -1.286389,
    "longitude": 36.817223,
    "start_date": "2023-01-01",
    "end_date": "2023-12-31",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "direct_normal_irradiance", # For Solar generation modeling
        "windspeed_10m",            # For Wind Generation Modelling
        "precipitation"
    ],
    "timezone": "Africa/Nairobi",
}

response = requests.get(url, params=params)
data = response.json()

# Load into DataFrame
df_weather = pd.DataFrame(data["hourly"])
df_weather['time'] = pd.to_datetime(df_weather['time'])
df_weather.set_index('time', inplace=True)

df_weather.head()

,temperature_2m,relative_humidity_2m,direct_normal_irradiance,windspeed_10m,precipitation
time,,,,,
2023-01-01 00:00:00,14.0,81,0.0,7.4,0.0
2023-01-01 01:00:00,14.5,75,0.0,8.9,0.0
2023-01-01 02:00:00,14.3,75,0.0,8.7,0.0
2023-01-01 03:00:00,13.9,78,0.0,7.4,0.0
2023-01-01 04:00:00,13.9,78,0.0,8.2,0.0


2. **Kaggle Kenyan Electricity Dataset**

In [9]:
from google.colab import files
import os
import pandas as pd

#upload my kaggle.json file when prompted
print ("Upload kaggle.json file")
uploaded = files.upload()

# Configure Kaggle API credentials
!mkdir -p ~/.Kaggle
!cp kaggle.json ~/.Kaggle/
!chmod 600 ~/.Kaggle/kaggle.json

# Downloading the kenyan electricity dataset directly from kaggle
!kaggle datasets download -d macmini62/kenyas-electricity-dataset --unzip

# Verify downloaded files
print ("\nDownloaded CSV files:")
print (os.listdir ('.'))

Upload kaggle.json file


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/macmini62/kenyas-electricity-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 3.04k/3.04k [00:00<00:00, 8.49MB/s]


Downloaded CSV files:
['.config', 'kenya-population.csv', 'kaggle.json', 'kenya-per-capita-electricity-generation.csv', 'kenya-electricity-generation.csv', 'kenya-electricity-demand.csv', 'sample_data']


In [12]:
# load kenya electricity generation & demand data
df_generation = pd.read_csv ('kenya-electricity-generation.csv')
df_demand = pd.read_csv ('kenya-electricity-demand.csv')
# df_consumption = pd.read_csv ('kenya-electricity-consumption.csv')

# inspect the generation dataset
print("--- Generation Data ---")
print (df_generation.head())
print ("\n--- Demand Data ---")
print (df_demand.head())
print ("\n--- Consumption Data ---")
# print (df_consumption.head())
#

--- Generation Data ---
  Country Code  Year  Coal  Fossil-fuels  Gas  Hydro  Low-carbon  Nuclear  \
0   Kenya  KEN  2000   0.0          2.57  0.0   1.31        1.74      0.0   
1   Kenya  KEN  2001   0.0          1.95  0.0   2.38        2.86      0.0   
2   Kenya  KEN  2002   0.0          1.20  0.0   3.09        3.95      0.0   
3   Kenya  KEN  2003   0.0          0.99  0.0   3.23        4.49      0.0   
4   Kenya  KEN  2004   0.0          1.82  0.0   2.84        4.34      0.0   

    Oil  Renewables  Solar  Solar-wind  Wind  Geothermal  \
0  2.57        1.74    0.0         0.0   0.0        0.43   
1  1.95        2.86    0.0         0.0   0.0        0.48   
2  1.20        3.95    0.0         0.0   0.0        0.86   
3  0.99        4.49    0.0         0.0   0.0        1.26   
4  1.82        4.34    0.0         0.0   0.0        1.50   

   Total-electricity-produced  
0                        4.31  
1                        4.81  
2                        5.15  
3                       